# 00 | Cadernos do TCC: roteiro e preparacao

Este caderno foi criado para organizar a reproducao, nao para gerar novos resultados cientificos. Os cadernos 01-05 recuperam as campanhas locais e o complemento MEDINA/R4; 06 compara os locais; 07-08 examinam imagens e a ficha humana; 09 reproduz graficos; 10 exporta os elementos aprovados e confere a atualizacao de R4. As analises historicas estao em outra pasta.

Nao e preciso refazer os treinamentos para consultar os arquivos. Nao execute todos os cadernos na mesma sessao: a separacao de ambientes e das campanhas faz parte do registro. Se a instalacao de bibliotecas pedir reinicio, reinicie e volte ao inicio do respectivo caderno.

## 1. Montar o Drive e localizar o pacote

O ZIP completo pode ser obtido na pasta de entrega. Ao instala-lo, nao sao substituidos arquivos diferentes ja existentes. Se a pasta ja foi extraida e contem README.md, esta etapa apenas a localiza.

In [ ]:
from pathlib import Path
from google.colab import drive, files
import zipfile, hashlib, json, shutil

drive.mount('/content/drive')
REPO = Path('/content/drive/MyDrive/mba_tcc/TCC_CellPainting_cadernos_repositorio')
PACOTE = None  # Caminho para TCC_CellPainting_repositorio_completo.zip, se ja estiver no Drive.

def sha(p):
    h = hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda: f.read(1048576), b''): h.update(b)
    return h.hexdigest()

if not (REPO / '04_fontes_e_referencia/recursos/MANIFESTO_SHA256.json').exists():
    if PACOTE is None:
        print('Selecione TCC_CellPainting_repositorio_completo.zip. Nenhum treinamento sera iniciado.')
        enviados = files.upload()
        candidatos = [Path(n) for n in enviados if n.endswith('.zip')]
        if len(candidatos) != 1: raise ValueError('Selecione exatamente um ZIP completo.')
        PACOTE = candidatos[0]
    with zipfile.ZipFile(PACOTE) as z:
        prefixo = 'TCC_CellPainting_repositorio/'
        plano = []
        for n in z.namelist():
            if n.endswith('/'): continue
            if not n.startswith(prefixo): raise ValueError('Estrutura de ZIP inesperada: ' + n)
            relativo = Path(n[len(prefixo):])
            destino = (REPO / relativo).resolve()
            if not destino.is_relative_to(REPO.resolve()): raise ValueError('Caminho inseguro.')
            blob = z.read(n)
            if destino.exists() and hashlib.sha256(blob).hexdigest() != sha(destino):
                if str(relativo).startswith('04_fontes_e_referencia/recursos/'):
                    raise FileExistsError('Recurso diferente preservado: ' + str(destino))
                print('Arquivo existente preservado, sem substituicao:', relativo)
                continue
            plano.append((destino, blob))
        for destino, blob in plano:
            destino.parent.mkdir(parents=True, exist_ok=True)
            if not destino.exists(): destino.write_bytes(blob)
print('Repositorio:', REPO)


## 2. Conferir recursos documentais

O manifesto confere os bytes entregues. Ele nao demonstra validade biologica nem substitui a conferencia dos dados de entrada.

In [ ]:
REC = REPO / '04_fontes_e_referencia/recursos'
manifesto = json.loads((REC / 'MANIFESTO_SHA256.json').read_text())
for relativo, esperado in manifesto.items():
    p = REC / relativo
    if not p.is_file() or sha(p) != esperado:
        raise RuntimeError('Recurso ausente ou alterado: ' + str(p))
print('Recursos conferidos:', len(manifesto))


## 3. Posicionar fontes documentais para os cadernos originais

Somente ZIPs pequenos de mapas/anotacoes sao copiados para os caminhos historicos esperados. Um arquivo ja existente com bytes distintos interrompe a etapa. Os perfis agregados, modelos, predições e imagens nao sao alterados. Essa compatibilidade de caminhos permite usar os cadernos originais sem reescrever sua logica.

In [ ]:
MBA = REPO.parent
plano = {
 'fontes_documentais_IMTM_HepG2_padronizado.zip': MBA / 'pipelines_IMTM_HepG2/fontes_documentais_IMTM_HepG2_padronizado.zip',
 'fontes_documentais_FMP_HepG2_padronizado.zip': MBA / 'pipelines_FMP_HepG2/fontes_documentais_FMP_HepG2_padronizado.zip',
 'fontes_documentais_MEDINA_HepG2.zip': MBA / 'pipelines_MEDINA_HepG2/fontes_documentais_MEDINA_HepG2.zip',
 'fontes_documentais_USC_HepG2.zip': MBA / 'pipelines_USC_HepG2/fontes_documentais_USC_HepG2.zip',
 'fontes_inspecao_biologica.zip': MBA / 'comparacao_4_locais_HepG2/inspecao_biologica_4_locais/Alda1_R1_R3_R4/fontes_inspecao_biologica.zip',
}
for nome, destino in plano.items():
    fonte = REC / 'fontes' / nome
    if destino.exists() and sha(destino) != sha(fonte): raise FileExistsError(destino)
for nome, destino in plano.items():
    destino.parent.mkdir(parents=True, exist_ok=True)
    if not destino.exists(): shutil.copy2(REC / 'fontes' / nome, destino)
print('Mapas e anotacoes disponiveis. Nenhum CSV experimental foi substituido.')


## 4. Verificar dados externos e escolher a etapa

Os dados brutos nao foram duplicados no repositorio. Cadernos 01-04 exigem CSVs agregados em suas pastas de origem. A integracao 08 exige tambem os ZIPs de leitura das quatro campanhas finais; nao substitua esses arquivos por outra execucao com metricas diferentes. Os cadernos 06, 09 e 10 podem trabalhar com os snapshots de referencia incluidos, sem retreinar. Os suplementares exigem as pastas historicas indicadas em cada um.

In [ ]:
RUNS = {'IMTM': 'IMTM_HepG2_20261003T185421_449156Z', 'FMP': 'FMP_HepG2_20261003T181504_724095Z', 'MEDINA': 'MEDINA_HepG2_20260930T235739_244794Z', 'USC': 'USC_HepG2_20261001T105954_572551Z'}
for site, run in RUNS.items():
    entrada = MBA / (site + '_HepG2')
    tipo = 'didatico_padronizado' if site in ['IMTM', 'FMP'] else 'didatico_generalizavel'
    leitura = MBA / ('resultados_' + site + '_HepG2') / tipo / (run + '_leitura.zip')
    print(site, '| CSVs:', len(list(entrada.glob('*_CP_Profiles_Aggregated.csv'))),
          '| pacote final:', 'presente' if leitura.is_file() else 'ausente')
print('Consulte 03_documentacao/dados_necessarios.md antes de executar uma etapa com fonte ausente.')
